# Model Comparison — Ship State Classification

Porównanie 6 modeli klasyfikacji stanów operacyjnych statku:
1. **Rule-based (FSM)** — maszyna stanów z progami kinematycznymi
2. **Random Forest** — klasyfikator na cechach okna czasowego
3. **XGBoost** — gradient boosting
4. **LightGBM** — gradient boosting (Microsoft)
5. **HMM** — Gaussian Hidden Markov Model
6. **LSTM** — bidirectional LSTM

In [ ]:
import sys
sys.path.insert(0, "..")

import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

STATE_COLORS = {"port_stay": "#2196F3", "anchor": "#FF9800", "adrift": "#F44336", "voyage": "#4CAF50"}
CLASS_NAMES = ["port_stay", "voyage", "anchor", "adrift"]

## 1. Wczytanie wyników wszystkich eksperymentów

In [ ]:
reports_dir = Path("../reports")

experiments = {}
for exp_dir in sorted(reports_dir.iterdir()):
    metrics_file = exp_dir / "metrics.json"
    if not metrics_file.exists():
        continue
    with metrics_file.open() as f:
        experiments[exp_dir.name] = json.load(f)

model_names = list(experiments.keys())
print(f"Loaded {len(experiments)} experiments: {model_names}")

## 2. Tabela porównawcza — Overall Accuracy

In [ ]:
rows = []
for name, metrics in experiments.items():
    rows.append({
        "Model": name,
        "Train Acc": metrics.get("train", {}).get("overall_accuracy", None),
        "Val Acc": metrics.get("val", {}).get("overall_accuracy", None),
        "Test Acc": metrics.get("test", {}).get("overall_accuracy", None),
    })

acc_df = pd.DataFrame(rows).set_index("Model")
acc_df.style.format("{:.4f}").highlight_max(axis=0, color="#c8e6c9").highlight_min(axis=0, color="#ffcdd2")

## 3. Wykres porównawczy — Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(model_names))
width = 0.25
splits = ["train", "val", "test"]
colors = ["#64B5F6", "#FFB74D", "#81C784"]

for i, (split, color) in enumerate(zip(splits, colors)):
    accs = [experiments[m].get(split, {}).get("overall_accuracy", 0) for m in model_names]
    bars = ax.bar(x + i * width, accs, width, label=split, color=color, alpha=0.85)
    for bar, acc in zip(bars, accs):
        if acc > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
                    f"{acc:.1%}", ha="center", va="bottom", fontsize=7, fontweight="bold")

ax.set_ylabel("Accuracy")
ax.set_title("Porównanie modeli — Overall Accuracy", fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels([m.replace("_gps_v1", "").replace("_", " ").title() for m in model_names], fontsize=9)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.15)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Per-class F1 Score — Validation & Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
states = ["port_stay", "voyage", "anchor", "adrift"]

for ax, split, title in zip(axes, ["val", "test"], ["Validation Set", "Test Set"]):
    x = np.arange(len(states))
    width = 0.8 / len(model_names)

    for i, model in enumerate(model_names):
        per_class = experiments[model].get(split, {}).get("per_class", {})
        f1s = [per_class.get(s, {}).get("f1", 0) for s in states]
        label = model.replace("_gps_v1", "").replace("_", " ")
        ax.bar(x + i * width, f1s, width, label=label, alpha=0.85)

    ax.set_ylabel("F1 Score")
    ax.set_title(f"Per-class F1 — {title}", fontsize=12)
    ax.set_xticks(x + width * len(model_names) / 2)
    ax.set_xticklabels(states)
    ax.legend(fontsize=7, loc="upper right")
    ax.set_ylim(0, 1.1)
    ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Confusion Matrices — najlepsze modele (Test Set)

In [ ]:
top_models = ["rule_based_gps_v1", "hmm_gps_v1", "lstm_gps_v1"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_name in zip(axes, top_models):
    cm_dict = experiments[model_name].get("test", {}).get("confusion_matrix", {})
    if not cm_dict:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        continue

    cm = pd.DataFrame(cm_dict).reindex(index=CLASS_NAMES, columns=CLASS_NAMES).fillna(0).astype(int)
    cm_norm = cm.div(cm.sum(axis=1), axis=0).fillna(0)

    sns.heatmap(cm_norm, annot=cm.values, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, vmin=0, vmax=1)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    label = model_name.replace("_gps_v1", "").replace("_", " ").title()
    test_acc = experiments[model_name]["test"]["overall_accuracy"]
    ax.set_title(f"{label}\n(test acc: {test_acc:.1%})", fontsize=11)

plt.suptitle("Confusion Matrices — Top 3 modele (Test Set)", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

## 6. Overfitting Analysis — Train vs Test Gap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

labels = [m.replace("_gps_v1", "").replace("_", " ").title() for m in model_names]
train_accs = [experiments[m]["train"]["overall_accuracy"] for m in model_names]
test_accs = [experiments[m]["test"]["overall_accuracy"] for m in model_names]
gaps = [t - te for t, te in zip(train_accs, test_accs)]

x = np.arange(len(model_names))
bars = ax.bar(x, gaps, color=["#ef5350" if g > 0.15 else "#66BB6A" for g in gaps], alpha=0.85)

for bar, gap, train, test in zip(bars, gaps, train_accs, test_accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"train={train:.1%}\ntest={test:.1%}", ha="center", va="bottom", fontsize=8)

ax.set_ylabel("Train - Test Accuracy Gap")
ax.set_title("Overfitting: Train-Test Gap (mniejszy = lepszy)", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.axhline(0.15, color="red", linestyle="--", alpha=0.5, label="overfitting threshold")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. HMM Transition Matrix

Macierz przejść HMM — prawdopodobieństwa przejść między stanami. Wysoka interpretowalność — widać naturalne wzorce dynamiki statku.

In [ ]:
tm_path = Path("../models/hmm_gps_v1/transition_matrix.json")
if tm_path.exists():
    with tm_path.open() as f:
        tm = json.load(f)

    states = ["adrift", "anchor", "port_stay", "voyage"]
    tm_array = np.array([[tm[s1][s2] for s2 in states] for s1 in states])

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(tm_array, annot=True, fmt=".4f", cmap="YlOrRd",
                xticklabels=states, yticklabels=states, ax=ax, vmin=0, vmax=1)
    ax.set_xlabel("To State")
    ax.set_ylabel("From State")
    ax.set_title("HMM Transition Matrix", fontsize=13)
    plt.tight_layout()
    plt.show()

    print("Interpretacja:")
    print("- Diagonala (~0.99): stany są bardzo stabilne (rzadkie przejścia)")
    print("- adrift -> voyage (2.2%): dryf kończy się wejściem w podróż")
    print("- voyage -> adrift (0.2%): sporadyczne spowolnienie podczas podróży")
else:
    print("HMM transition matrix not found — run HMM experiment first")

## 8. Podsumowanie i wnioski

### Ranking modeli (wg Test Accuracy)

| # | Model | Test Acc | Komentarz |
|---|-------|---------|-----------|
| 1 | **Rule-based (FSM)** | **98.3%** | Najstabilniejszy, wiedza domenowa |
| 2 | HMM | 83.0% | Interpretowalny, macierz przejść |
| 3 | LSTM | 73.6% | Early stopping, za mało danych |
| 4 | Random Forest | 50.5% | Silny overfitting |
| 5 | XGBoost | 50.0% | Silny overfitting |
| 6 | LightGBM | 38.2% | Najsilniejszy overfitting |

### Kluczowe wnioski

1. **Wiedza domenowa > ML przy małym datasecie** — rule-based z fizycznymi progami (SOG, spread) generalizuje najlepiej
2. **Overfitting klasycznych ML** — 99.8% train vs 38-50% test to klasyczny przypadek zbyt małego datasetu (21 epizodów)
3. **HMM — najlepszy ML model** — wykorzystuje naturalną dynamikę Markowa stanów statku, macierz przejść daje interpretowalność na konferencji
4. **Adrift najtrudniejszy** — rzadki stan, nieliczne epizody, żaden model nie wykrywa go konsekwentnie na teście
5. **Potrzeba więcej danych** — z pełnym datasetem (więcej statków, dłuższe rejsy) ML powinien znacząco się poprawić